# Baseline 3: Hệ thống cải tiến (Enhanced RAG) - Trực quan hóa Luồng dữ liệu

## 1. Sơ đồ Luồng Dữ liệu (Data Flow)
Để hiểu rõ cách thức hoạt động của RAG nâng cao, hãy theo dõi luồng biến đổi của dữ liệu:
1. **Query Gốc** -> Trích xuất năm tài chính & Lọc cứng danh sách tài liệu (`active_keys`).
2. **Query Gốc** -> Mở rộng từ đồng nghĩa -> `expanded_query`.
3. **Retrieval Song Song**:
   - **BM25**: Tìm trên `expanded_query` giới hạn trong `active_keys` -> Trả về xếp hạng BM25.
   - **Dense HNSW**: Tìm trên `original_query` giới hạn trong `active_keys` -> Trả về xếp hạng HNSW.
4. **RRF (Rank Fusion)**: Gộp 2 danh sách xếp hạng dựa trên vị trí -> Trả về danh sách `candidates`.
5. **Cross-Encoder**: Xếp hạng lại (Rerank) các `candidates` -> Trả về kết quả tối ưu nhất.

## 2. Khởi tạo Dữ liệu & Khai báo Thư viện

In [ ]:
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

corpus = {
    "doc1": "Apple Net sales in fiscal year 2023 were $383,285 million.",
    "doc2": "Apple payments for acquisition of property plant and equipment were $10,959 million.",
    "doc3": "Amazon net sales were $574,785 million in fiscal year 2023.",
    "doc4": "Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.",
    "doc5": "NVIDIA net income was $29,760 million in fiscal year 2024."
}

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s$]', '', text)
    return text.split()

tokenized_docs = {k: tokenize(v) for k, v in corpus.items()}
vocabulary = sorted(list(set(word for doc in tokenized_docs.values() for word in doc)))

print(f"[Log] Khởi tạo thành công. Từ vựng gồm {len(vocabulary)} từ.")

## 3. Tải Mô hình

In [ ]:
print("[Log] Đang tải mô hình Bi-Encoder (BGE) và Cross-Encoder (MiniLM)...")
bi_encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("[SUCCESS] Đã nạp xong models!")

## 4. Minh họa Chi tiết từng Bước (Step-by-Step)
Ta chạy thử nghiệm với câu hỏi: **"Amazon capital expenditures 2023"**

In [ ]:
query = "Amazon capital expenditures 2023"
print(f"INPUT GỐC: '{query}'")

### Bước 4.1: Year Routing
**Mục tiêu:** Trích xuất năm và lọc chỉ giữ lại tài liệu của năm đó.

In [ ]:
# Trích xuất năm
match = re.search(r'\b(20\d{2})\b', query)
target_year = int(match.group(1)) if match else None
print(f"[Process] Trích xuất năm: {target_year}")

# Lọc danh sách tài liệu đang hoạt động (active_keys)
active_keys = [k for k, text in corpus.items() if not target_year or str(target_year) in text]

print(f"\nOUTPUT BƯỚC 4.1:")
print(f"  - target_year = {target_year}")
print(f"  - active_keys = {active_keys}")
print("  (doc5 thuộc năm 2024 đã bị loại bỏ)")

### Bước 4.2: Query Expansion
**Mục tiêu:** Mở rộng từ đồng nghĩa tài chính để hỗ trợ bộ tìm kiếm từ khóa.

In [ ]:
SYNONYMS_DICT = {
    "capital expenditures": ["purchases of property and equipment", "capex"],
    "net sales": ["revenue", "sales"],
    "net income": ["profit", "net earnings"]
}

expanded_query = query
query_lower = query.lower()
for key_phrase, synonyms in SYNONYMS_DICT.items():
    if key_phrase in query_lower:
        expanded_query = query + " " + " ".join(synonyms)
        print(f"[Process] Khớp cụm từ '{key_phrase}' -> Bổ sung: {synonyms}")
        break

print(f"\nOUTPUT BƯỚC 4.2:")
print(f"  - expanded_query = '{expanded_query}'")

### Bước 4.3: Tìm kiếm Song song (Parallel Retrieval)
#### 1. Xếp hạng bằng BM25 (trên `expanded_query` & `active_keys`)
In ma trận tính toán điểm chi tiết từng từ.

In [ ]:
# Tính toán BM25 IDF tĩnh
bm25_idf = {}
for word in vocabulary:
    n_qi = sum(1 for tokens in tokenized_docs.values() if word in tokens)
    bm25_idf[word] = np.log((len(corpus) - n_qi + 0.5) / (n_qi + 0.5) + 1.0)

doc_lengths = {k: len(v) for k, v in tokenized_docs.items()}
avgdl = sum(doc_lengths.values()) / len(doc_lengths)

k1, b = 1.2, 0.75
q_tokens = tokenize(expanded_query)
bm25_scores = {}

print("=== MA TRẬN CHI TIẾT TÍNH ĐIỂM TỪNG TỪ KHÓA (BM25) ===")
for k in active_keys:
    tokens = tokenized_docs[k]
    doc_len = len(tokens)
    score = 0.0
    print(f"\n* Tài liệu {k} (độ dài {doc_len}):")
    for word in q_tokens:
        if word in vocabulary:
            tf = tokens.count(word)
            idf = bm25_idf[word]
            if tf > 0:
                term_score = idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (doc_len / avgdl)))
                score += term_score
                print(f"  - Từ '{word}': tf={tf}, idf={idf:.4f} -> Điểm = {term_score:.4f}")
    bm25_scores[k] = score
    print(f"  => Tổng điểm BM25 cho {k} = {score:.4f}")

bm25_ranked = [doc for doc, sc in sorted(bm25_scores.items(), key=lambda x: x[1], reverse=True) if sc > 0]
print(f"\nOUTPUT BM25 RANKING: {bm25_ranked}")

#### 2. Xếp hạng bằng Dense HNSW (trên `query` gốc & `active_keys`)
In ma trận tính toán Cosine Similarity chi tiết.

In [ ]:
active_texts = [corpus[k] for k in active_keys]
active_embeddings = bi_encoder.encode(active_texts, normalize_embeddings=True)
query_vector = bi_encoder.encode([query], normalize_embeddings=True)[0]

hnsw_scores = {}
print("=== MA TRẬN COSINE SIMILARITY DENSE ===")
for i, k in enumerate(active_keys):
    doc_vector = active_embeddings[i]
    cosine_sim = np.dot(query_vector, doc_vector)
    hnsw_scores[k] = cosine_sim
    print(f"  - {k} vs Query: Cosine = {cosine_sim:.4f} (Trích 3 chiều đầu: q={query_vector[:3].round(3)}, d={doc_vector[:3].round(3)})")

hnsw_ranked = [doc for doc, sc in sorted(hnsw_scores.items(), key=lambda x: x[1], reverse=True)]
print(f"\nOUTPUT HNSW RANKING: {hnsw_ranked}")

### Bước 4.4: Gộp Thứ Hạng bằng RRF (Reciprocal Rank Fusion)
*   **Input:** `bm25_ranked` và `hnsw_ranked` (đầu ra từ Bước 4.3).
*   **Hành động:** Tính điểm RRF dựa trên vị trí xếp hạng để gộp kết quả.
*   **Output:** Bảng điểm RRF chi tiết.

In [ ]:
k_constant = 60
rrf_scores = {}
doc_track = {}

for rank, doc_name in enumerate(bm25_ranked):
    rank_1based = rank + 1
    score = 1.0 / (k_constant + rank_1based)
    rrf_scores[doc_name] = rrf_scores.get(doc_name, 0.0) + score
    doc_track[doc_name] = {'bm25_rank': rank_1based, 'bm25_score': score, 'hnsw_rank': 'N/A', 'hnsw_score': 0.0}

for rank, doc_name in enumerate(hnsw_ranked):
    rank_1based = rank + 1
    score = 1.0 / (k_constant + rank_1based)
    rrf_scores[doc_name] = rrf_scores.get(doc_name, 0.0) + score
    if doc_name in doc_track:
        doc_track[doc_name]['hnsw_rank'] = rank_1based
        doc_track[doc_name]['hnsw_score'] = score
    else:
        doc_track[doc_name] = {'bm25_rank': 'N/A', 'bm25_score': 0.0, 'hnsw_rank': rank_1based, 'hnsw_score': score}

sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

rows = []
for doc_name, total_score in sorted_rrf:
    t = doc_track[doc_name]
    rows.append({
        'Tài liệu': doc_name,
        'Hạng BM25': t['bm25_rank'],
        'Điểm RRF BM25': f"1/(60+{t['bm25_rank']})" if t['bm25_rank'] != 'N/A' else '0',
        'Hạng HNSW': t['hnsw_rank'],
        'Điểm RRF HNSW': f"1/(60+{t['hnsw_rank']})" if t['hnsw_rank'] != 'N/A' else '0',
        'Tổng Điểm RRF': f"{total_score:.6f}"
    })

df_rrf = pd.DataFrame(rows)
print("=== MA TRẬN GỘP THỨ HẠNG RRF ===")
print(df_rrf.to_string(index=False))

candidates = [doc_name for doc_name, _ in sorted_rrf]
print(f"\nOUTPUT BƯỚC 4.4 (Candidates): {candidates}")

### Bước 4.5: Xếp hạng lại bằng Cross-Encoder (Reranking)
*   **Input:** `query` gốc và danh sách `candidates` (đầu ra từ Bước 4.4).
*   **Hành động:** Ghép cặp chéo và đưa vào mô hình Cross-Encoder để tính điểm tương quan sâu.
*   **Output:** Xếp hạng cuối cùng của hệ thống.

In [ ]:
pairs = [[query, corpus[cand]] for cand in candidates]
print("=== INPUT CỦA CROSS-ENCODER ===")
for i, pair in enumerate(pairs):
    print(f"  - Cặp {i+1}: Query = '{pair[0]}' | Doc = '{pair[1][:45]}...'")

ce_scores = cross_encoder.predict(pairs)
reranked_results = sorted(zip(candidates, ce_scores), key=lambda x: x[1], reverse=True)

print("\n=== OUTPUT CUỐI CÙNG (XẾP HẠNG CROSS-ENCODER) ===")
for rank, (doc_name, score) in enumerate(reranked_results):
    print(f"  Hạng {rank+1}: {doc_name} | Score = {score:.4f} | Nội dung: {corpus[doc_name]}")

## 5. Thử nghiệm Khử nhiễu Thời gian
Chúng ta thử nghiệm với câu hỏi: **"NVIDIA net income was what in fiscal year 2023?"** để thấy hiệu quả của lớp Year Routing.

In [ ]:
query_temporal = "NVIDIA net income was what in fiscal year 2023?"
print(f"=== TRUY VẤN: '{query_temporal}' ===")

# Bước 1: Trích xuất năm
match_temp = re.search(r'\b(20\d{2})\b', query_temporal)
year_temp = int(match_temp.group(1)) if match_temp else None
print(f"  - Trích xuất năm: {year_temp}")

# Bước 2: Lọc tài liệu theo năm
active_keys_temp = [k for k, text in corpus.items() if not year_temp or str(year_temp) in text]
print(f"  - Danh sách tài liệu được giữ lại: {active_keys_temp}")
print("    (doc5 chứa dữ liệu năm 2024 của NVIDIA bị loại bỏ hoàn toàn)")

# Bước 3: Tìm kiếm ngữ nghĩa trên tập tài liệu đã lọc
temp_texts = [corpus[k] for k in active_keys_temp]
temp_embeddings = bi_encoder.encode(temp_texts, normalize_embeddings=True)
query_temp_vector = bi_encoder.encode([query_temporal], normalize_embeddings=True)[0]

temp_scores = {}
for i, k in enumerate(active_keys_temp):
    sim = np.dot(query_temp_vector, temp_embeddings[i])
    temp_scores[k] = sim

ranked_temp = sorted(temp_scores.items(), key=lambda x: x[1], reverse=True)
print("\n=== KẾT QUẢ TÌM KIẾM DENSE CUỐI CÙNG (KHÔNG CÒN NHIỄU NĂM 2024) ===")
for rank, (doc_name, score) in enumerate(ranked_temp):
    print(f"  Hạng {rank+1}: {doc_name} | Score Cosine = {score:.4f} | Nội dung: {corpus[doc_name]}")